# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engyelgamal18/flyrank-ml-internship-engy/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

My research question is: Which content pages should be reviewed first for possible CTR improvement?
The goal is to create a priority ranking that can help editors decide which pages may need attention based on search performance data.  

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [1]:
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

march_path = hf_hub_download(
    repo_id ="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

df = pd.read_parquet(
    march_path,
    columns=[
        "report_date",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
)

df["report_date"] = pd.to_datetime(df["report_date"])

val_df = df[df["report_date"] > "2026.03.24"].copy()

val_summary = (
    val_df.groupby("content_hash_id", as_index=False)
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks","sum"),
        gsc_avg_position=("gsc_avg_position", "mean")
    )
)

val_summary["ctr"] = np.where(
    val_summary["gsc_impressions"] > 0,
    val_summary["gsc_clicks"] / val_summary["gsc_impressions"],
    0
)
# Week 4 baseline
baseline_df = val_summary.copy()

baseline_df["expected_ctr"] = np.select(
    [
        baseline_df["gsc_avg_position"] <= 3,
        baseline_df["gsc_avg_position"] <= 10,
        baseline_df["gsc_avg_position"] <= 20
    ],
    [0.004576, 0.003473, 0.002770],
    default=0.001289
)

baseline_df["baseline_score"] = (
    baseline_df["gsc_impressions"]*
    (baseline_df["expected_ctr"] - baseline_df["ctr"]).clip(lower=0)
)

baseline_df = baseline_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_df["baseline_rank"] = range(1, len(baseline_df) + 1)

# Week 5 ranking
ranking_df = val_summary.copy()

ranking_df["gsc_avg_position"] = ranking_df["gsc_avg_position"].fillna(100)

ranking_df["priority_score"] = (
    ranking_df["gsc_impressions"] *
    (1- ranking_df["ctr"]) *
    ranking_df["gsc_avg_position"]
)

ranking_df = ranking_df.sort_values(
    "priority_score", ascending=False
).reset_index(drop=True)

ranking_df["new_rank"]= range(1, len(ranking_df) + 1)

print("Validation rows:", len(val_df))
print("Pages ranked:", len(ranking_df))
print("Validation dates:", val_df["report_date"].min(), "to", val_df["report_date"].max())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Validation rows: 2292889
Pages ranked: 331436
Validation dates: 2026-03-25 00:00:00 to 2026-03-31 00:00:00


I used the March 2026 data from the fact content daily performance table in the FlyRank internship warehouse. I used data from March 1 to March 24 for the earlier period and march 25 to March 31 for validation. I used search signals such as impressions, clicks, CTR and average position because they can help identify pages that may need CTR review. I excluded client names, URLS, private search queries and trend fields because they were not needed for the ranking and could create privacy or data leakage risks.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

I used a ranking analysis for CTR review. The main features were impressions, CTR and average search position. I assumed that pages with meaningful search visibility but weaker CTR may be useful candidates for editorial review. The priority score was used as a proxy to rank pages that may need review rathar than as a true label of content quality. The Week 4 baseline used imoressions and the gap between expected CTR and actual CTR. I used a time aware split with March 1-24 as the earlier period and March 25-31 for validation. I did not use client information, private queries or label derived trend fields to reduce privacy and data leakage risks.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The week 4 baseline and the new ranking were compared using the same validation period March 25 -31,2026. The new ranking changed the priority of several pages. For example one page moved from baseline rank 136 to new rank 1 while another moved from rank 300 to rank 9. Some pages with a baseline score of zero also moved much higher in the new ranking.

In [2]:
comparison_df = baseline_df[
    ["content_hash_id", "baseline_rank", "baseline_score"]
].merge(
    ranking_df[
        ["content_hash_id", "new_rank", "priority_score"]
    ],
    on="content_hash_id",
    how="inner"
)

comparison_df = comparison_df.sort_values("new_rank")

comparison_df.head(10)

,content_hash_id,baseline_rank,baseline_score,new_rank,priority_score
135,content_73aa61dcedebbf30,136,32.891941,1,1.332174e+06
262144,content_36e53e9c707674fc,262145,0.000000,2,1.254553e+06
224221,content_66288edeb93b7c4f,224222,0.000000,3,1.093095e+06
85,content_661a7734f691bef5,86,39.978661,4,1.048987e+06
248,content_82e35c4845e6c391,249,25.750213,5,1.047175e+06
921,content_3f9e8f387f3fe7e7,922,13.288891,6,9.753288e+05
161839,content_fa84f5976d5fe3c1,161840,0.000000,7,8.189318e+05
217,content_a3a1317f7c2bc3dd,218,27.351555,8,7.945984e+05
299,content_ab91e088440ace78,300,23.774316,9,7.704454e+05
1837,content_136c4bf04b07b778,1838,8.791570,10,7.686408e+05


The Week 4 baseline and the new ranking were compared using the same validation period: March 25-31/2026. The new method changed the priority of many pages. For example one page moved from baseline rank 136 to new rank 1 . Another page moved from baseline rank 300 to new rank 9. Some pages with baseline score of xero also moved to the top of the new ranking. This shows that the new method priotrizes some pages differently from the baseline and may surface additional candidates for review. These results are directional. The ranking is a decision support tool and does not prove that the new method is always better.

## 5. Limitations

*What this work cannot claim.*

This analysis has some limitations. The priority ranking is based only on search performance signals such as impressions, CTR and average position. It does not include content quality, conversions or business value. The results are observational and directional not casual. A high priority score does not prove that changing a page will improve its CTR. The ranking should be used as a guide to help editors decide which pages to review first. The analysis is limited to March 2026 data so the ranking should be reviewed again when newer data becomes avaliable.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
top_recommendations = ranking_df[
[
    "content_hash_id",
    "gsc_impressions",
    "ctr",
    "gsc_avg_position",
    "priority_score",
    "new_rank"
]
].head(10)

top_recommendations

,content_hash_id,gsc_impressions,ctr,gsc_avg_position,priority_score,new_rank
0,content_73aa61dcedebbf30,27069,0.000074,49.217629,1.332174e+06,1
1,content_36e53e9c707674fc,39542,0.001467,31.773706,1.254553e+06,2
2,content_66288edeb93b7c4f,79987,0.005276,13.738387,1.093095e+06,3
3,content_661a7734f691bef5,39549,0.000278,26.531121,1.048987e+06,4
4,content_82e35c4845e6c391,34717,0.000547,30.179682,1.047175e+06,5
5,content_3f9e8f387f3fe7e7,19619,0.000612,49.743909,9.753288e+05,6
6,content_fa84f5976d5fe3c1,20912,0.001387,39.215236,8.189318e+05,7
7,content_a3a1317f7c2bc3dd,21995,0.000045,36.127962,7.945984e+05,8
8,content_ab91e088440ace78,18444,0.000000,41.772143,7.704454e+05,9
9,content_136c4bf04b07b778,16130,0.000744,47.688351,7.686408e+05,10


The highest ranked pages should be reviewed first because the ranking identifies pages with combinations of meaningful impressions, low CTR and weaker average search position. Editors can use these signals as reason codes for review, low CTR can prompt a review of titles and snipples while weaker search position can prompt a broader review of content and search performance. The ranking determines review priority not whether a page should automatically be changed. All recommended actions require human review.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
artifact_table = comparison_df.head(10)[
    ["new_rank", "baseline_rank"]
].copy()

artifact_table["Rank Improvement"] = (
    artifact_table["baseline_rank"] -artifact_table["new_rank"]
)


artifact_table = artifact_table.rename(columns={
    "new_rank": "New Rank",
    "baseline_rank": "Week 4 Baseline Rank"
})


artifact_table

,New Rank,Week 4 Baseline Rank,Rank Improvement
135,1,136,135
262144,2,262145,262143
224221,3,224222,224219
85,4,86,82
248,5,249,244
921,6,922,916
161839,7,161840,161833
217,8,218,210
299,9,300,291
1837,10,1838,1828


In [9]:
import plotly.express as px

chart_df = artifact_table.copy()
chart_df["Page"] = ["Page " + str(i) for i in range(1, len(chart_df) + 1)]

fig = px.bar(
    chart_df,
    x="Rank Improvement",
    y="Page",
    orientation="h",
    title="Rank Improvement for Top 10 Recommended Pages"
)

fig.update_layout(
    xaxis_title="Position Improved vs Week 4 Baseline",
    yaxis_title="Top 10 Recommended Pages",
    yaxis=dict(autorange="reversed")
)

fig.show()

The chart shows that several pages moved substantially higher in the new ranking compared with Week 4 baseline. The size of the changes varies widely across pages showing that the new method priortizes some review candidates very differently. These results are directional and should be used to support human review rather than as evidence that the new ranking is always better.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [*] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [*] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


## ML-12 closing

### 5-minutes demo outline

1. Introduce thequestion: Which content pages should be reviewed first for possible CTR improvement?
2. Explain the data: March 2026 search performance data using impressions, CTR and average search position.
3. Explain the method: Compare Week 4 baseline with new priority ranking using a time aware validation split.
4. Show the results: Some pages moved much higher in the new ranking including baseline rank 136 to new rank 1.
5. End with the action: Use the ranking as a decision support tool to help editors decide which pages to review first.

Completed my FlyRank ML internship capstone on CTR/Engagement opportunity scoring. I analyzed March 2026 search performance data and built a priority ranking using impressions, CTR and average position to identify pages that may be worth reviewing first. The project uses time aware validation and compars the new ranking with a baseline. The final output is designed as a decision support tool for content review.

### 3- sentences employer _facing summary

I designed a repeatable ranking analysis to priortize content pages for CTR review using real search performance data. The workflow combines impressions, CTR and validates the ranking aganist a baseline using time aware split. The final output gives editors a practical review queue while keeping the results directional and focused on  decision support.